In [54]:
# Lab 2
import numpy as np
import pandas as pd

file = pd.read_csv("../data/YearPredictionMSD.txt", header=None, delimiter=",").dropna()

train_df = file.iloc[:463715]
test_df = file.iloc[463715:]

y_train = train_df.iloc[:, 0].values
X_train = train_df.iloc[:, 1:].values

y_test = test_df.iloc[:, 0].values
X_test = test_df.iloc[:, 1:].values

A_train = np.c_[np.ones(X_train.shape[0]), X_train]
A_test = np.c_[np.ones(X_test.shape[0]), X_test]

f_min = X_train.min(axis=0)
f_max = X_train.max(axis=0)

denominator_f = np.where((f_max - f_min) == 0, 1, (f_max - f_min))

X_train_scaled = (X_train - f_min) / denominator_f
X_test_scaled = (X_test - f_min) / denominator_f

A_train_scaled = np.c_[np.ones(X_train_scaled.shape[0]), X_train_scaled]
A_test_scaled = np.c_[np.ones(X_test_scaled.shape[0]), X_test_scaled]


y_min = y_train.min()
y_max = y_train.max()

denominator_y = 1 if (y_max - y_min) == 0 else (y_max - y_min)

y_train_scaled = (y_train - y_min) / denominator_y
y_test_scaled = (y_test - y_min) / denominator_y

def fit_ols(A, y):  # ordinary least squares
    M = A.T @ A
    v = A.T @ y
    w = np.linalg.solve(M, v)
    return w

# Lab 11
## Ex 3


In [55]:
import time
from scipy.linalg import eigh

In [56]:
H_scaled = A_train_scaled.T @ A_train_scaled

eigenvalues, _ = eigh(H_scaled)
lambda_min = eigenvalues[0]
lambda_max = eigenvalues[-1]

alpha = 1 / (lambda_max + lambda_min)

In [57]:
print(f"Najmniejsza wartość własna: {lambda_min:.4f}")
print(f"Największa wartość własna: {lambda_max:.4f}")
print(f"Optymalna stała ucząca (alpha): {alpha:.8f}")

Najmniejsza wartość własna: 4.5766
Największa wartość własna: 9823507.3344
Optymalna stała ucząca (alpha): 0.00000010


In [58]:
w_gd = np.zeros(A_train_scaled.shape[1])
max_iterations = 20_000
epsilon = 1e-5

In [ ]:
start_time_gd = time.time()

for i in range(max_iterations):
    # Obliczenie gradientu: 2 * A^T * (A * w - y)
    error_vector = A_train_scaled @ w_gd - y_train_scaled
    gradient = 2 * A_train_scaled.T @ error_vector

    w_gd_new = w_gd - alpha * gradient

    if np.linalg.norm(w_gd_new - w_gd) < epsilon:
        w_gd = w_gd_new
        print(f"Zbieżność osiągnięta po {i + 1} iteracjach.")
        break

    w_gd = w_gd_new
else:
    print("Osiągnięto maksymalną liczbę iteracji bez zbieżności.")

end_time_gd = time.time()
gd_time = end_time_gd - start_time_gd

In [ ]:
start_time_ols = time.time()
w_ols = fit_ols(A_train_scaled, y_train_scaled)
end_time_ols = time.time()
ols_time = end_time_ols - start_time_ols

In [ ]:
# Predykcje dla GD
y_pred_gd_scaled = A_test_scaled @ w_gd
y_pred_gd = y_pred_gd_scaled * denominator_y + y_min
error_gd = np.average(np.abs(y_pred_gd - y_test))

# Predykcje dla OLS
y_pred_ols_scaled = A_test_scaled @ w_ols
y_pred_ols = y_pred_ols_scaled * denominator_y + y_min
error_ols = np.average(np.abs(y_pred_ols - y_test))

In [ ]:
print("-" * 40)
print(f"Błąd (MAE) - OLS:              {error_ols:.4f}")
print(f"Błąd (MAE) - Gradient Descent: {error_gd:.4f}")
print(f"Czas obliczeń - OLS:              {ols_time:.4f} s")
print(f"Czas obliczeń - Gradient Descent: {gd_time:.4f} s")

----------------------------------------
Błąd (MAE) - OLS:              6.8005
Błąd (MAE) - Gradient Descent: 76.3491
Czas obliczeń - OLS:              0.1969 s
Czas obliczeń - Gradient Descent: 187.4837 s


## Zoptymalizowane zadanie 3

In [ ]:
# Prekomputacja macierzy (optymalizacja obliczeń)
M = A_train_scaled.T @ A_train_scaled
v = A_train_scaled.T @ y_train_scaled

start_time_gd_optimized = time.time()

for i in range(max_iterations):
    # 2 * A^T * (A * w - y) -> 2 * (M @ w - v)
    gradient = 2 * (M @ w_gd - v)

    w_gd_new = w_gd - alpha * gradient

    if np.linalg.norm(w_gd_new - w_gd) < epsilon:
        w_gd = w_gd_new
        print(f"Zbieżność osiągnięta po {i + 1} iteracjach.")
        break

    w_gd = w_gd_new
else:
    print("Osiągnięto maksymalną liczbę iteracji bez zbieżności.")

end_time_gd_optimized = time.time()
gd_time_optimized = end_time_gd_optimized - start_time_gd_optimized

# Predykcje dla Gradient Descent
preds_gd_scaled = A_test_scaled @ w_gd
preds_gd_adjusted = preds_gd_scaled * denominator_y + y_min
error_gd_optimized = np.average(np.abs(preds_gd_adjusted - y_test))

print("-" * 40)
print(f"Błąd (MAE) - Gradient Descent:    {error_gd:.4f}")
print(f"Czas obliczeń - Gradient Descent: {gd_time:.4f} s")
print(f"Błąd (MAE) - Gradient Descent Optimized:    {error_gd_optimized:.4f}")
print(f"Czas obliczeń - Gradient Descent Optimized: {gd_time_optimized:.4f} s")

Osiągnięto maksymalną liczbę iteracji bez zbieżności.
----------------------------------------
Błąd (MAE) - Gradient Descent:    76.1949
Czas obliczeń - Gradient Descent: 0.0261 s
Błąd (MAE) - Gradient Descent Optimized:    75.5998
Czas obliczeń - Gradient Descent Optimized: 0.0227 s
